## I. Recursive Feature Elimination (RFE) Pipeline for Multi-Response Prediction

Overview

This pipeline performs Recursive Feature Elimination (RFE) using Random Forest to identify the most important predictors for sugarcane yield and quality metrics. It supports multiple response variables and includes comprehensive cross-validation evaluation.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, RFECV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import json
import gc
import psutil
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# MEMORY MANAGEMENT UTILITIES
# ============================================================================

def print_memory_usage(stage=""):
    """Print current memory usage"""
    process = psutil.Process(os.getpid())
    memory_mb = process.memory_info().rss / 1024 / 1024
    print(f"   [Memory] {stage}: {memory_mb:.2f} MB")

def clean_memory():
    """Force garbage collection and memory cleanup"""
    gc.collect()
    
def aggressive_memory_cleanup():
    """Aggressive memory cleanup with multiple garbage collection passes"""
    for _ in range(3):
        gc.collect()
    print("   [Memory] Aggressive cleanup completed")

# ============================================================================
# DEFINE AVAILABLE RESPONSE VARIABLES (Same as original)
# ============================================================================

SUGAR_CONTENT_METRICS = {
    "brix_percent": "Brix (%)",
    "pol_juice_percent": "Pol juice (%)",
    "pol_cane_percent": "Pol cane (%)",
    "purity_percent": "Purity (%)",
    "glucose_cane_percent": "Glucose (%)",
    "hugot_sugar": "Hugot sugar (%)"
}

YIELD_PARAMETERS = {
    "tc_per_ha_actual": "Cane yield (t ha⁻¹)",
    "tc_per_ha_per_week": "Cane yield per week (t ha⁻¹/week)",
    "tc_per_ha_52_weeks": "Cane yield 52 weeks (t ha⁻¹)",
    "fiber_percent": "Fiber in Cane (%)",
}

ALL_RESPONSE_VARIABLES = {**SUGAR_CONTENT_METRICS, **YIELD_PARAMETERS}

# ============================================================================
# DATA LOADING AND PREPARATION
# ============================================================================

def load_and_prepare_data_for_rfe(file_path, response_variable, all_predictors=None, 
                                   categorical_vars=None):
    """Load CSV data and prepare it for RFE with specified response variable"""
    print(f"Starting RFE Analysis for Response Variable: {response_variable}")
    print(f"Display name: {ALL_RESPONSE_VARIABLES.get(response_variable, response_variable)}")
    print("="*80)

    df = pd.read_csv(file_path)
    
    # Check if response variable exists
    if response_variable not in df.columns:
        raise ValueError(f"Response variable '{response_variable}' not found in dataset!")
    
    # Drop ALL other response variables (keep only target)
    all_response_cols = list(ALL_RESPONSE_VARIABLES.keys())
    cols_to_drop = [col for col in all_response_cols 
                    if col != response_variable and col in df.columns]
    
    if cols_to_drop:
        print(f"Dropping other response variables: {cols_to_drop}")
        df = df.drop(columns=cols_to_drop)

    # Handle predictors
    if all_predictors is not None:
        print(f"Total predictors provided: {len(all_predictors)}")
        
        year_column = None
        for col in ['campaign_year', 'Harvest_Year', 'year']:
            if col in df.columns:
                year_column = col
                break

        columns_to_keep = [response_variable] + all_predictors
        if year_column:
            columns_to_keep.append(year_column)
        if categorical_vars:
            columns_to_keep.extend(categorical_vars)

        # Check for missing predictors
        missing_predictors = [col for col in all_predictors if col not in df.columns]
        if missing_predictors:
            print(f"WARNING: Predictors not found in dataset: {missing_predictors}")
            all_predictors = [col for col in all_predictors if col in df.columns]
            print(f"Continuing with {len(all_predictors)} available predictors")
        
        if categorical_vars:
            missing_categorical = [col for col in categorical_vars if col not in df.columns]
            if missing_categorical:
                raise ValueError(f"Categorical variables not found: {missing_categorical}")

        # Keep only relevant columns
        columns_to_keep = [response_variable] + all_predictors
        if year_column:
            columns_to_keep.append(year_column)
        if categorical_vars:
            columns_to_keep.extend(categorical_vars)
        
        df = df[columns_to_keep]
        print(f"Dataset columns retained: {len(df.columns)}")

    # Remove rows with missing target values
    initial_count = len(df)
    df = df.dropna(subset=[response_variable])
    final_count = len(df)
    
    if initial_count - final_count > 0:
        print(f"Dropped {initial_count - final_count} observations with missing {response_variable}")

    # Identify year column
    year_column = None
    for col in ['campaign_year', 'Harvest_Year', 'year']:
        if col in df.columns:
            year_column = col
            break

    samples = df.copy()
    encoders = {}

    # Encode year if present
    if year_column:
        unique_years = sorted(samples[year_column].unique())
        print(f"\nYears available: {unique_years}")
        year_counts = samples[year_column].value_counts().sort_index()
        for year, count in year_counts.items():
            print(f"   {year}: {count} samples")
        
        year_encoder = LabelEncoder()
        samples[f'{year_column}_encoded'] = year_encoder.fit_transform(samples[year_column])
        encoders[year_column] = year_encoder
    else:
        unique_years = ["All data"]
        print("No year column found")

    # Encode categorical variables
    if categorical_vars:
        print(f"\nEncoding categorical variables: {categorical_vars}")
        for cat_var in categorical_vars:
            if cat_var in samples.columns:
                cat_encoder = LabelEncoder()
                samples[f'{cat_var}_encoded'] = cat_encoder.fit_transform(samples[cat_var].astype(str))
                encoders[cat_var] = cat_encoder
                print(f"   {cat_var}: {len(cat_encoder.classes_)} unique values")

    # Fill missing values in numeric columns
    numeric_cols = samples.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if samples[col].isnull().any():
            samples[col] = samples[col].fillna(samples[col].median())

    print(f"\nTotal dataset shape: {samples.shape}")
    print(f"{response_variable} statistics:\n{samples[response_variable].describe()}")

    return samples, encoders, unique_years, year_column

def prepare_features_for_rfe(samples, encoders, year_column, response_variable, 
                             all_predictors=None, categorical_vars=None):
    """Prepare feature list for RFE"""
    TARGET = response_variable

    if all_predictors is not None:
        # Use provided predictors
        available_predictors = [col for col in all_predictors if col in samples.columns]
        FEATURES = available_predictors.copy()
        
        # Add encoded categorical variables
        if categorical_vars:
            for cat_var in categorical_vars:
                encoded_col = f'{cat_var}_encoded'
                if encoded_col in samples.columns:
                    FEATURES.append(encoded_col)
        
        print(f"\nUsing {len(available_predictors)} numeric predictors")
        if categorical_vars:
            print(f"Added {len(categorical_vars)} encoded categorical variables")
    else:
        # Use all numeric columns except target and excluded columns
        numeric_cols = samples.select_dtypes(include=[np.number]).columns.tolist()
        exclude_cols = [response_variable]
        if year_column:
            exclude_cols.extend([year_column, f'{year_column}_encoded'])
        if categorical_vars:
            exclude_cols.extend(categorical_vars)
        # Exclude all other response variables
        all_response_cols = list(ALL_RESPONSE_VARIABLES.keys())
        exclude_cols.extend([col for col in all_response_cols if col != response_variable])
        FEATURES = [col for col in numeric_cols if col not in exclude_cols]

    print(f"Total features for RFE: {len(FEATURES)}")
    return FEATURES, TARGET

# ============================================================================
# RFE FUNCTIONS
# ============================================================================

def perform_rfe(X, y, n_features_to_select=25, n_estimators=800, random_state=42):
    """
    Perform Recursive Feature Elimination using Random Forest
    Similar to: RFE(estimator=RandomForestRegressor(...), n_features_to_select=25, step=1)
    """
    print(f"\n=== PERFORMING RFE ===")
    print(f"Initial features: {X.shape[1]}")
    print(f"Target features: {n_features_to_select}")
    print(f"Random Forest estimators: {n_estimators}")
    print_memory_usage("Before RFE")
    
    # Create Random Forest estimator with memory-efficient settings
    estimator = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=None,
        max_features='sqrt',
        min_samples_split=2,
        min_samples_leaf=1,
        bootstrap=True,
        oob_score=True,
        n_jobs=-1,
        random_state=random_state,
        verbose=0,
        warm_start=False  # Ensure no memory accumulation
    )
    
    # Scale features
    print("\nScaling features...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    clean_memory()
    
    # Perform RFE
    print("\nRunning RFE (this may take a while)...")
    print_memory_usage("Before RFE fit")
    
    selector = RFE(
        estimator=estimator,
        n_features_to_select=n_features_to_select,
        step=1,
        verbose=1  # Show progress
    )
    
    selector.fit(X_scaled, y)
    print_memory_usage("After RFE fit")
    
    # Get selected features
    selected_features = X.columns[selector.support_].tolist()
    feature_rankings = pd.DataFrame({
        'Feature': X.columns,
        'Selected': selector.support_,
        'Ranking': selector.ranking_
    }).sort_values('Ranking')
    
    # Clean up
    del X_scaled
    clean_memory()
    
    print(f"\nRFE Complete!")
    print(f"Selected {len(selected_features)} features")
    print_memory_usage("After RFE complete")
    
    return selector, selected_features, feature_rankings, scaler

def perform_rfecv(X, y, min_features=10, max_features=None, step=1, cv=5, 
                  n_estimators=800, random_state=42):
    """
    Perform RFE with Cross-Validation to automatically determine optimal number of features
    """
    if max_features is None:
        max_features = X.shape[1]
    
    print(f"\n=== PERFORMING RFECV ===")
    print(f"Initial features: {X.shape[1]}")
    print(f"Min features to select: {min_features}")
    print(f"Cross-validation folds: {cv}")
    print(f"Random Forest estimators: {n_estimators}")
    print_memory_usage("Before RFECV")
    
    # Create Random Forest estimator with memory-efficient settings
    estimator = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=None,
        max_features='sqrt',
        min_samples_split=2,
        min_samples_leaf=1,
        bootstrap=True,
        oob_score=True,
        n_jobs=-1,
        random_state=random_state,
        verbose=0,
        warm_start=False
    )
    
    # Scale features
    print("\nScaling features...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    clean_memory()
    
    # Perform RFECV
    print("\nRunning RFECV (this may take a while)...")
    print_memory_usage("Before RFECV fit")
    
    selector = RFECV(
        estimator=estimator,
        step=step,
        cv=cv,
        scoring='r2',
        min_features_to_select=min_features,
        n_jobs=-1,
        verbose=1  # Show progress
    )
    
    selector.fit(X_scaled, y)
    print_memory_usage("After RFECV fit")
    
    # Get selected features
    selected_features = X.columns[selector.support_].tolist()
    optimal_n_features = selector.n_features_
    
    feature_rankings = pd.DataFrame({
        'Feature': X.columns,
        'Selected': selector.support_,
        'Ranking': selector.ranking_
    }).sort_values('Ranking')
    
    # Clean up
    del X_scaled
    clean_memory()
    
    print(f"\nRFECV Complete!")
    print(f"Optimal number of features: {optimal_n_features}")
    print(f"Best CV score: {selector.cv_results_['mean_test_score'][optimal_n_features - min_features]:.4f}")
    print_memory_usage("After RFECV complete")
    
    return selector, selected_features, feature_rankings, scaler, optimal_n_features

# ============================================================================
# CROSS-VALIDATION EVALUATION
# ============================================================================

def calculate_rpd_rpiq(y_true, y_pred):
    """Calculate RPD and RPIQ metrics"""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    sd = np.std(y_true)
    q75, q25 = np.percentile(y_true, [75, 25])
    iqr = q75 - q25
    
    rpd = sd / rmse if rmse > 0 else 0
    rpiq = iqr / rmse if rmse > 0 else 0
    
    return rpd, rpiq

def evaluate_selected_features_cv(X, y, selected_features, n_folds=5, n_estimators=800, 
                                   random_state=42):
    """
    Evaluate the selected features using k-fold cross-validation
    """
    print(f"\n=== EVALUATING SELECTED FEATURES WITH {n_folds}-FOLD CV ===")
    print(f"Number of selected features: {len(selected_features)}")
    print_memory_usage("Before CV evaluation")
    
    X_selected = X[selected_features].copy()
    
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    
    fold_results = []
    all_predictions = []
    all_actuals = []
    
    estimator = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=None,
        max_features='sqrt',
        min_samples_split=2,
        min_samples_leaf=1,
        bootstrap=True,
        oob_score=True,
        n_jobs=-1,
        random_state=random_state,
        verbose=0,
        warm_start=False
    )
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_selected), 1):
        print(f"\n   Processing Fold {fold}/{n_folds}...")
        print_memory_usage(f"Fold {fold} start")
        
        X_train, X_val = X_selected.iloc[train_idx].copy(), X_selected.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()
        
        # Scale features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        
        # Train and predict
        estimator.fit(X_train_scaled, y_train)
        y_pred = estimator.predict(X_val_scaled)
        
        # Calculate metrics
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        r2 = r2_score(y_val, y_pred)
        mae = mean_absolute_error(y_val, y_pred)
        rpd, rpiq = calculate_rpd_rpiq(y_val, y_pred)
        
        fold_results.append({
            'fold': fold,
            'rmse': rmse,
            'r2': r2,
            'mae': mae,
            'rpd': rpd,
            'rpiq': rpiq,
            'n_samples': len(y_val)
        })
        
        all_predictions.extend(y_pred)
        all_actuals.extend(y_val)
        
        # Clean up fold data
        del X_train, X_val, X_train_scaled, X_val_scaled, y_train, y_val, y_pred, scaler
        clean_memory()
        
        print(f"   Fold {fold}: R² = {r2:.4f}, RMSE = {rmse:.4f}, RPD = {rpd:.4f}")
        print_memory_usage(f"Fold {fold} end")
    
    # Calculate overall metrics
    overall_rmse = np.sqrt(mean_squared_error(all_actuals, all_predictions))
    overall_r2 = r2_score(all_actuals, all_predictions)
    overall_mae = mean_absolute_error(all_actuals, all_predictions)
    overall_rpd, overall_rpiq = calculate_rpd_rpiq(all_actuals, all_predictions)
    
    # Calculate mean and std across folds
    fold_rmse = [f['rmse'] for f in fold_results]
    fold_r2 = [f['r2'] for f in fold_results]
    fold_mae = [f['mae'] for f in fold_results]
    fold_rpd = [f['rpd'] for f in fold_results]
    fold_rpiq = [f['rpiq'] for f in fold_results]
    
    cv_results = {
        'fold_results': fold_results,
        'mean_rmse': np.mean(fold_rmse),
        'std_rmse': np.std(fold_rmse),
        'mean_r2': np.mean(fold_r2),
        'std_r2': np.std(fold_r2),
        'mean_mae': np.mean(fold_mae),
        'std_mae': np.std(fold_mae),
        'mean_rpd': np.mean(fold_rpd),
        'std_rpd': np.std(fold_rpd),
        'mean_rpiq': np.mean(fold_rpiq),
        'std_rpiq': np.std(fold_rpiq),
        'overall_rmse': overall_rmse,
        'overall_r2': overall_r2,
        'overall_mae': overall_mae,
        'overall_rpd': overall_rpd,
        'overall_rpiq': overall_rpiq
    }
    
    # Clean up
    del X_selected, all_predictions, all_actuals
    aggressive_memory_cleanup()
    
    print(f"\n=== CV RESULTS SUMMARY ===")
    print(f"Mean CV R²: {cv_results['mean_r2']:.4f} ± {cv_results['std_r2']:.4f}")
    print(f"Mean CV RMSE: {cv_results['mean_rmse']:.4f} ± {cv_results['std_rmse']:.4f}")
    print(f"Mean CV RPD: {cv_results['mean_rpd']:.4f} ± {cv_results['std_rpd']:.4f}")
    print(f"Mean CV RPIQ: {cv_results['mean_rpiq']:.4f} ± {cv_results['std_rpiq']:.4f}")
    print(f"Overall R²: {cv_results['overall_r2']:.4f}")
    print(f"Overall RMSE: {cv_results['overall_rmse']:.4f}")
    print(f"Overall RPD: {cv_results['overall_rpd']:.4f}")
    print(f"Overall RPIQ: {cv_results['overall_rpiq']:.4f}")
    print_memory_usage("After CV evaluation")
    
    return cv_results

def get_feature_importance_from_selected(X, y, selected_features, n_estimators=800, 
                                        random_state=42):
    """Get feature importance for selected features"""
    print(f"\n=== CALCULATING FEATURE IMPORTANCE ===")
    print_memory_usage("Before feature importance")
    
    X_selected = X[selected_features].copy()
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_selected)
    
    # Train Random Forest on all data
    estimator = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=None,
        max_features='sqrt',
        min_samples_split=2,
        min_samples_leaf=1,
        bootstrap=True,
        oob_score=True,
        n_jobs=-1,
        random_state=random_state,
        verbose=0,
        warm_start=False
    )
    
    estimator.fit(X_scaled, y)
    
    # Get feature importance
    importance_df = pd.DataFrame({
        'Feature': selected_features,
        'Importance': estimator.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    # Clean up
    del X_selected, X_scaled, scaler
    clean_memory()
    
    print(f"\nTop 15 Most Important Features:")
    for idx, row in importance_df.head(15).iterrows():
        print(f"   {row['Feature']:<40}: {row['Importance']:.4f}")
    
    print_memory_usage("After feature importance")
    
    return importance_df, estimator

# ============================================================================
# VISUALIZATION
# ============================================================================

def plot_rfe_results(feature_rankings, selected_features, output_dir, response_variable):
    """Create visualizations for RFE results"""
    print("\n=== CREATING VISUALIZATIONS ===")
    
    os.makedirs(os.path.join(output_dir, 'plots'), exist_ok=True)
    
    # 1. Feature ranking distribution
    plt.figure(figsize=(12, 6))
    colors = ['green' if x else 'red' for x in feature_rankings['Selected']]
    plt.bar(range(len(feature_rankings)), feature_rankings['Ranking'], color=colors)
    plt.xlabel('Feature Index')
    plt.ylabel('Ranking')
    plt.title(f'RFE Feature Rankings - {response_variable}')
    plt.axhline(y=1, color='black', linestyle='--', label='Selected (Rank=1)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'plots', f'rfe_rankings_{response_variable}.png'), dpi=300)
    plt.close('all')
    clean_memory()
    
    print(f"   Saved: rfe_rankings_{response_variable}.png")

def plot_rfecv_results(selector, output_dir, response_variable):
    """Plot RFECV results showing optimal number of features"""
    print("\n=== CREATING RFECV VISUALIZATION ===")
    
    os.makedirs(os.path.join(output_dir, 'plots'), exist_ok=True)
    
    # Plot number of features vs CV score
    plt.figure(figsize=(12, 6))
    n_features = range(selector.min_features_to_select, 
                      len(selector.cv_results_['mean_test_score']) + selector.min_features_to_select)
    plt.plot(n_features, selector.cv_results_['mean_test_score'], marker='o')
    plt.xlabel('Number of Features')
    plt.ylabel('Cross-Validation R² Score')
    plt.title(f'RFECV - Optimal Number of Features - {response_variable}')
    plt.axvline(x=selector.n_features_, color='red', linestyle='--', 
                label=f'Optimal: {selector.n_features_} features')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'plots', f'rfecv_scores_{response_variable}.png'), dpi=300)
    plt.close('all')
    clean_memory()
    
    print(f"   Saved: rfecv_scores_{response_variable}.png")

def plot_feature_importance(importance_df, output_dir, response_variable, top_n=25):
    """Plot feature importance for selected features"""
    print("\n=== CREATING FEATURE IMPORTANCE PLOT ===")
    
    os.makedirs(os.path.join(output_dir, 'plots'), exist_ok=True)
    
    # Plot top N features
    plt.figure(figsize=(12, 8))
    top_features = importance_df.head(top_n)
    plt.barh(range(len(top_features)), top_features['Importance'])
    plt.yticks(range(len(top_features)), top_features['Feature'])
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.title(f'Top {top_n} Feature Importance - {response_variable}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'plots', f'feature_importance_{response_variable}.png'), dpi=300)
    plt.close('all')
    clean_memory()
    
    print(f"   Saved: feature_importance_{response_variable}.png")

# ============================================================================
# SAVE RESULTS
# ============================================================================

def save_rfe_results(selector, selected_features, feature_rankings, cv_results, 
                    importance_df, encoders, response_variable, output_dir, 
                    use_rfecv=False, final_model=None, final_scaler=None):
    """Save all RFE results and models"""
    print("\n=== SAVING RESULTS ===")
    
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'results'), exist_ok=True)
    
    # Save selector object
    selector_filename = f'rfe_selector_{response_variable}.pkl'
    joblib.dump({
        'selector': selector,
        'encoders': encoders,
        'selected_features': selected_features
    }, os.path.join(output_dir, 'models', selector_filename))
    print(f"   Saved: {selector_filename}")
    
    # Save final model if provided
    if final_model is not None and final_scaler is not None:
        model_filename = f'final_model_{response_variable}.pkl'
        joblib.dump({
            'model': final_model,
            'scaler': final_scaler,
            'selected_features': selected_features,
            'encoders': encoders
        }, os.path.join(output_dir, 'models', model_filename))
        print(f"   Saved: {model_filename}")
    
    # Save feature rankings
    feature_rankings.to_csv(
        os.path.join(output_dir, 'results', f'feature_rankings_{response_variable}.csv'), 
        index=False
    )
    print(f"   Saved: feature_rankings_{response_variable}.csv")
    
    # Save selected features list
    selected_features_df = pd.DataFrame({'Selected_Features': selected_features})
    selected_features_df.to_csv(
        os.path.join(output_dir, 'results', f'selected_features_{response_variable}.csv'), 
        index=False
    )
    print(f"   Saved: selected_features_{response_variable}.csv")
    
    # Save CV results
    cv_results_df = pd.DataFrame(cv_results['fold_results'])
    cv_results_df.to_csv(
        os.path.join(output_dir, 'results', f'cv_fold_results_{response_variable}.csv'), 
        index=False
    )
    print(f"   Saved: cv_fold_results_{response_variable}.csv")
    
    # Save feature importance
    importance_df.to_csv(
        os.path.join(output_dir, 'results', f'feature_importance_{response_variable}.csv'), 
        index=False
    )
    print(f"   Saved: feature_importance_{response_variable}.csv")
    
    # Save summary JSON
    summary = {
        'response_variable': response_variable,
        'response_variable_display': ALL_RESPONSE_VARIABLES.get(response_variable, response_variable),
        'method': 'RFECV' if use_rfecv else 'RFE',
        'n_features_selected': len(selected_features),
        'selected_features': selected_features,
        'cv_performance': {
            'mean_r2': float(cv_results['mean_r2']),
            'std_r2': float(cv_results['std_r2']),
            'mean_rmse': float(cv_results['mean_rmse']),
            'std_rmse': float(cv_results['std_rmse']),
            'mean_rpd': float(cv_results['mean_rpd']),
            'std_rpd': float(cv_results['std_rpd']),
            'mean_rpiq': float(cv_results['mean_rpiq']),
            'std_rpiq': float(cv_results['std_rpiq']),
            'overall_r2': float(cv_results['overall_r2']),
            'overall_rmse': float(cv_results['overall_rmse']),
            'overall_rpd': float(cv_results['overall_rpd']),
            'overall_rpiq': float(cv_results['overall_rpiq'])
        },
        'top_10_features': importance_df.head(10)[['Feature', 'Importance']].to_dict('records')
    }
    
    with open(os.path.join(output_dir, f'rfe_summary_{response_variable}.json'), 'w') as f:
        json.dump(summary, f, indent=4)
    print(f"   Saved: rfe_summary_{response_variable}.json")
    
    # Save encoder information
    if encoders:
        encoder_info = {}
        for var_name, encoder in encoders.items():
            encoder_info[var_name] = {
                'classes': [str(c) for c in encoder.classes_]
            }
        with open(os.path.join(output_dir, f'encoders_info_{response_variable}.json'), 'w') as f:
            json.dump(encoder_info, f, indent=4)
        print(f"   Saved: encoders_info_{response_variable}.json")
    
    print(f"\nAll results saved to: {output_dir}")

# ============================================================================
# MAIN RFE PIPELINE
# ============================================================================

def run_rfe_pipeline(file_path, response_variable, all_predictors=None, 
                    categorical_vars=None, n_features_to_select=25, 
                    use_rfecv=False, min_features_rfecv=10, 
                    n_estimators=800, n_folds=5, random_state=42,
                    base_output_dir=None):
    """
    Main RFE pipeline with flexible response variable selection
    
    Parameters:
    -----------
    file_path : str
        Path to the CSV file
    response_variable : str
        Name of the response variable to predict
    all_predictors : list, optional
        List of all predictor variable names to consider
    categorical_vars : list, optional
        List of categorical variable names
    n_features_to_select : int, default=25
        Number of features to select (used if use_rfecv=False)
    use_rfecv : bool, default=False
        If True, use RFECV to automatically determine optimal number of features
    min_features_rfecv : int, default=10
        Minimum number of features for RFECV
    n_estimators : int, default=800
        Number of trees in Random Forest
    n_folds : int, default=5
        Number of cross-validation folds
    random_state : int, default=42
        Random state for reproducibility
    base_output_dir : str, optional
        Base directory for saving outputs
    
    Returns:
    --------
    dict : Dictionary containing results and outputs
    """
    
    # Validate response variable
    if response_variable not in ALL_RESPONSE_VARIABLES:
        raise ValueError(f"Response variable '{response_variable}' not recognized!\n"
                        f"Available options: {list(ALL_RESPONSE_VARIABLES.keys())}")
    
    # Create output directory
    if base_output_dir is None:
        base_output_dir = os.getcwd()
    
    method_name = 'rfecv' if use_rfecv else f'rfe_{n_features_to_select}'
    output_dir = os.path.join(base_output_dir, f'{method_name}_results_{response_variable}')
    
    if categorical_vars:
        output_dir += '_with_categorical'
    
    os.makedirs(output_dir, exist_ok=True)
    
    print("\n" + "="*80)
    print(f"RFE PIPELINE FOR: {response_variable}")
    print(f"Display name: {ALL_RESPONSE_VARIABLES[response_variable]}")
    print(f"Method: {'RFECV (auto-select)' if use_rfecv else f'RFE (select {n_features_to_select})'}")
    print("="*80)
    print(f"Output directory: {output_dir}\n")
    print_memory_usage("Pipeline start")
    
    # Load and prepare data
    print("\nStep 1: Loading and preparing data...")
    samples, encoders, unique_years, year_column = load_and_prepare_data_for_rfe(
        file_path, response_variable, all_predictors, categorical_vars
    )
    clean_memory()
    print_memory_usage("After data loading")
    
    # Prepare features
    print("\nStep 2: Preparing features...")
    FEATURES, TARGET = prepare_features_for_rfe(
        samples, encoders, year_column, response_variable, all_predictors, categorical_vars
    )
    
    X = samples[FEATURES].copy()
    y = samples[TARGET].copy()
    
    # Free up memory from samples dataframe
    del samples
    clean_memory()
    
    print(f"\nDataset prepared:")
    print(f"   Samples: {len(X)}")
    print(f"   Features: {len(FEATURES)}")
    print(f"   Target range: {y.min():.2f} - {y.max():.2f}")
    print_memory_usage("After feature preparation")
    
    # Perform RFE or RFECV
    if use_rfecv:
        print("\nStep 3: Performing RFECV (automatic feature selection)...")
        selector, selected_features, feature_rankings, scaler, optimal_n = perform_rfecv(
            X, y, min_features=min_features_rfecv, step=1, cv=n_folds,
            n_estimators=n_estimators, random_state=random_state
        )
        aggressive_memory_cleanup()
        plot_rfecv_results(selector, output_dir, response_variable)
    else:
        print(f"\nStep 3: Performing RFE (selecting {n_features_to_select} features)...")
        selector, selected_features, feature_rankings, scaler = perform_rfe(
            X, y, n_features_to_select=n_features_to_select,
            n_estimators=n_estimators, random_state=random_state
        )
        aggressive_memory_cleanup()
    
    print_memory_usage("After RFE/RFECV")
    
    # Display selected features
    print(f"\n=== SELECTED FEATURES ({len(selected_features)}) ===")
    for i, feature in enumerate(selected_features, 1):
        print(f"   {i:2d}. {feature}")
    
    # Evaluate with cross-validation
    print("\nStep 4: Evaluating selected features with cross-validation...")
    cv_results = evaluate_selected_features_cv(
        X, y, selected_features, n_folds=n_folds,
        n_estimators=n_estimators, random_state=random_state
    )
    aggressive_memory_cleanup()
    print_memory_usage("After CV evaluation")
    
    # Get feature importance
    print("\nStep 5: Calculating feature importance...")
    importance_df, final_model = get_feature_importance_from_selected(
        X, y, selected_features, n_estimators=n_estimators, random_state=random_state
    )
    clean_memory()
    print_memory_usage("After feature importance")
    
    # Create visualizations
    print("\nStep 6: Creating visualizations...")
    plot_rfe_results(feature_rankings, selected_features, output_dir, response_variable)
    plot_feature_importance(importance_df, output_dir, response_variable, 
                          top_n=min(25, len(selected_features)))
    aggressive_memory_cleanup()
    print_memory_usage("After visualizations")
    
    # Train final model on selected features
    print("\nStep 7: Training final model on all data...")
    X_selected = X[selected_features].copy()
    final_scaler = StandardScaler()
    X_scaled = final_scaler.fit_transform(X_selected)
    final_model.fit(X_scaled, y)
    
    # Clean up
    del X_selected, X_scaled, X, y
    aggressive_memory_cleanup()
    print_memory_usage("After final model training")
    
    # Save results
    print("\nStep 8: Saving results...")
    save_rfe_results(
        selector, selected_features, feature_rankings, cv_results,
        importance_df, encoders, response_variable, output_dir,
        use_rfecv=use_rfecv, final_model=final_model, final_scaler=final_scaler
    )
    
    # Final summary
    print("\n" + "="*80)
    print("RFE PIPELINE COMPLETE!")
    print("="*80)
    print(f"\nResponse Variable: {response_variable}")
    print(f"Method: {'RFECV' if use_rfecv else 'RFE'}")
    print(f"Features Selected: {len(selected_features)}")
    print(f"\nPerformance Metrics:")
    print(f"   Mean CV R²: {cv_results['mean_r2']:.4f} ± {cv_results['std_r2']:.4f}")
    print(f"   Mean CV RMSE: {cv_results['mean_rmse']:.4f} ± {cv_results['std_rmse']:.4f}")
    print(f"   Mean CV RPD: {cv_results['mean_rpd']:.4f} ± {cv_results['std_rpd']:.4f}")
    print(f"   Overall R²: {cv_results['overall_r2']:.4f}")
    print(f"   Overall RMSE: {cv_results['overall_rmse']:.4f}")
    print(f"   Overall RPD: {cv_results['overall_rpd']:.4f}")
    print(f"\nTop 5 Most Important Features:")
    for idx, row in importance_df.head(5).iterrows():
        print(f"   {row['Feature']:<40}: {row['Importance']:.4f}")
    print(f"\nAll results saved to: {output_dir}")
    print_memory_usage("Pipeline complete")
    print("="*80)
    
    return {
        'selector': selector,
        'selected_features': selected_features,
        'feature_rankings': feature_rankings,
        'cv_results': cv_results,
        'importance_df': importance_df,
        'final_model': final_model,
        'final_scaler': final_scaler,
        'encoders': encoders,
        'output_dir': output_dir,
        'response_variable': response_variable
    }

# ============================================================================
# USAGE EXAMPLES
# ============================================================================

if __name__ == "__main__":
    
    # Define all your predictors (complete list)
    all_predictors = ['pH_complete','Exchangeable_Calcium_meq_100g_complete','Exchangeable_Magnesium_meq_100g_complete','Exchangeable_Potassium_meq_100g_complete','Clay_percent_complete','Fine_Silt_percent_complete','Coarse_Silt_percent_complete','Fine_Sand_percent_complete','Coarse_Sand_percent_complete',
                      'Total_Nitrogen_per_mille_complete','Organic_Carbon_per_mille_complete','Mg_K_Ratio_complete','Mg_Ca_Ratio_complete','growing_season_rainfall_mm','temp_mini_mean','temp_maxi_mean','temp_moy_mean','ampl_mean','h_mini_mean','h_maxi_mean','h_moy_mean','h_6h_mean','h_12h_mean','h_16h_mean','h_24h_mean',
                      'soil_temp_6h_mean','soil_temp_16h_mean','radiation_ERA_total','evaporation_total','CWD','CDD','SDII','R10','R20','Rx1','Rx5','R95p','R99p','HD',
                      'TNn','TNx','TXn','TXx','Available_P2O5_mg_kg_complete','Compaction_Index_complete','Electrical_Conductivity_mmho_cm_complete','Exchangeable_Sodium_meq_100g_complete']


    
    # Categorical variables
    categorical_features = []
    
    # File path
    data_file = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv'
    
    # Base output directory
    base_output = 'C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/RFE_Results'
    
    print("="*80)
    print("RFE FEATURE SELECTION PIPELINE - MULTIPLE RESPONSE VARIABLES")
    print("="*80)
    print("\nAvailable response variables:")
    print("\nSUGAR CONTENT METRICS:")
    for key, label in SUGAR_CONTENT_METRICS.items():
        print(f"  - {key}: {label}")
    print("\nYIELD PARAMETERS:")
    for key, label in YIELD_PARAMETERS.items():
        print(f"  - {key}: {label}")
    print("="*80 + "\n")
    
    print_memory_usage("Initial")
    
    try:
        # Example 1: RFE with fixed number of features (25) for brix_percent
        # Using fewer trees (400 instead of 800) to reduce memory usage
        print("\n### EXAMPLE 1: RFE (25 features) for brix_percent ###\n")
        results_brix_rfe = run_rfe_pipeline(
            file_path=data_file,
            response_variable='hugot_sugar',
            all_predictors=all_predictors,
            categorical_vars=categorical_features,
            n_features_to_select=25,
            use_rfecv=False,
            n_estimators=400,  
            n_folds=5,
            random_state=42,
            base_output_dir=base_output
        )
        
        # Clean up between runs
        del results_brix_rfe
        aggressive_memory_cleanup()
        print("\n[CLEANUP] Memory cleaned between pipelines")
        print_memory_usage("After Example 1")
        
        # Example 2: RFECV (automatic feature selection) for hugot_sugar
        #print("\n\n### EXAMPLE 2: RFECV (auto-select) for hugot_sugar ###\n")
        #results_hugot_rfecv = run_rfe_pipeline(
        #    file_path=data_file,
        #    response_variable='hugot_sugar',
        #    all_predictors=all_predictors,
        #    categorical_vars=categorical_features,
        #    use_rfecv=True,
        #    min_features_rfecv=10,
        #    n_estimators=400, 
        #    n_folds=5,
        #    random_state=42,
        #    base_output_dir=base_output
        #)
        
        # Clean up between runs
        #del results_hugot_rfecv
        #aggressive_memory_cleanup()
        #print("\n[CLEANUP] Memory cleaned between pipelines")
        #print_memory_usage("After Example 2")
        
        # Example 3: RFE for cane yield
        print("\n\n### EXAMPLE 3: RFE (25 features) for tc_per_ha_actual ###\n")
        results_yield_rfe = run_rfe_pipeline(
            file_path=data_file,
            response_variable='purity_percent',
            all_predictors=all_predictors,
            categorical_vars=categorical_features,
            n_features_to_select=25,
            use_rfecv=False,
            n_estimators=400, 
            n_folds=5,
            random_state=42,
            base_output_dir=base_output
        )
        
        # Final cleanup
        del results_yield_rfe
        aggressive_memory_cleanup()
        print_memory_usage("After Example 3")
        

        
        print("\nAvailable response variables:")
        for key in ALL_RESPONSE_VARIABLES.keys():
            print(f"  - '{key}'")
        print("="*80)
        
    except MemoryError as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()
        
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()

RFE FEATURE SELECTION PIPELINE - MULTIPLE RESPONSE VARIABLES

Available response variables:

SUGAR CONTENT METRICS:
  - brix_percent: Brix (%)
  - pol_juice_percent: Pol juice (%)
  - pol_cane_percent: Pol cane (%)
  - purity_percent: Purity (%)
  - glucose_cane_percent: Glucose (%)
  - hugot_sugar: Hugot sugar (%)

YIELD PARAMETERS:
  - tc_per_ha_actual: Cane yield (t ha⁻¹)
  - tc_per_ha_per_week: Cane yield per week (t ha⁻¹/week)
  - tc_per_ha_52_weeks: Cane yield 52 weeks (t ha⁻¹)
  - fiber_percent: Fiber in Cane (%)

   [Memory] Initial: 79.81 MB

### EXAMPLE 1: RFE (25 features) for brix_percent ###


RFE PIPELINE FOR: hugot_sugar
Display name: Hugot sugar (%)
Method: RFE (select 25)
Output directory: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/RFE_Results\rfe_25_results_hugot_sugar

   [Memory] Pipeline start: 79.95 MB

Step 1: Loading and preparing data...
Starting RFE Analysis for Response Variable: hugot_sugar
Display name: Hugot sugar (%)
Drop

## II. Variance Inflation Factor (VIF) Analysis for Multicollinearity Detection

Overview

This pipeline performs Variance Inflation Factor (VIF) analysis to identify and address multicollinearity among selected predictor variables. It evaluates features obtained from feature selection methods and recommends which features to retain for modeling.

Key Features

✅ Multicollinearity Detection - Calculate VIF for all features to identify redundant predictors
✅ Categorical Variable Support - Handles categorical features via one-hot encoding
✅ Automated Thresholding - Classifies features as Low (VIF<5), Moderate (5≤VIF<10), or High (VIF≥10)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# FONT SETUP
# ============================================================================

available_fonts = [f.name for f in fm.fontManager.ttflist]
if 'Arial' in available_fonts:
    font_family = 'Arial'
elif 'Helvetica' in available_fonts:
    font_family = 'Helvetica'
elif 'DejaVu Sans' in available_fonts:
    font_family = 'DejaVu Sans'
elif 'Liberation Sans' in available_fonts:
    font_family = 'Liberation Sans'
else:
    font_family = 'sans-serif'

plt.rcParams.update({
    'font.family': font_family,
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 12,
    'figure.titlesize': 16,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'axes.axisbelow': True,
    'axes.edgecolor': '#333333',
    'text.color': '#333333',
    'axes.labelcolor': '#333333',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'text.usetex': False,
    'mathtext.default': 'regular',
})

# ============================================================================
# LANGUAGE SUPPORT
# ============================================================================

TRANSLATIONS = {
    'en': {
        'title': 'VIF Analysis',
        'xlabel': 'VIF Value',
        'ylabel': 'VIF Value',
        'feature_label': 'Feature',
        'threshold_low': 'VIF = 5',
        'threshold_high': 'VIF = 10',
        'category_low': 'Low (VIF < 5)',
        'category_moderate': 'Moderate (5 ≤ VIF < 10)',
        'category_high': 'High (VIF ≥ 10)',
        'categorical_marker': 'Categorical',
        'vif_stats_title': 'VIF Statistics',
        'mean': 'Mean',
        'median': 'Median',
        'max': 'Max',
        'n_features': 'Features',
        'loading': 'Loading data from',
        'preparing': 'Preparing features for VIF analysis',
        'calculating': 'Calculating VIF values',
        'complete': 'VIF Analysis Complete',
        'saved': 'Saved',
    },
    'fr': {
        'title': 'Analyse VIF',
        'xlabel': 'Valeur VIF',
        'ylabel': 'Valeur VIF',
        'feature_label': 'Variable',
        'threshold_low': 'VIF = 5',
        'threshold_high': 'VIF = 10',
        'category_low': 'Faible (VIF < 5)',
        'category_moderate': 'Modéré (5 ≤ VIF < 10)',
        'category_high': 'Élevé (VIF ≥ 10)',
        'categorical_marker': 'Catégorique',
        'vif_stats_title': 'Statistiques VIF',
        'mean': 'Moyenne',
        'median': 'Médiane',
        'max': 'Max',
        'n_features': 'Variables',
        'loading': 'Chargement des données depuis',
        'preparing': 'Préparation des variables pour analyse VIF',
        'calculating': 'Calcul des valeurs VIF',
        'complete': 'Analyse VIF Terminée',
        'saved': 'Enregistré',
    }
}

# ============================================================================
# FEATURE DISPLAY NAMES
# ============================================================================

FEATURE_LABELS = {
    'pH_complete': {
        'en': r'pH$_{\mathrm{pre\text{-}plant}}$',
        'fr': r'pH$_{\mathrm{avant\text{ }plant.}}$'
    },
    'Exchangeable_Calcium_meq_100g_complete': {'en': r'Ca exch. (meq 100g$^{-1}$)', 'fr': r'Ca éch. (meq 100g$^{-1}$)'},
    'Exchangeable_Potassium_meq_100g_complete': {'en': r'K exch. (meq 100g$^{-1}$)', 'fr': r'K éch. (meq 100g$^{-1}$)'},
    'Exchangeable_Sodium_meq_100g_complete': {'en': r'Na exch. (meq 100g$^{-1}$)', 'fr': r'Na éch. (meq 100g$^{-1}$)'},
    'Available_P2O5_mg_kg_complete': {'en': r'P$_2$O$_5$ avail. (mg kg$^{-1}$)', 'fr': r'P$_2$O$_5$ disp. (mg kg$^{-1}$)'},
    'Total_Nitrogen_per_mille_complete': {'en': 'N total (‰)', 'fr': 'N total (‰)'},
    'Electrical_Conductivity_mmho_cm_complete': {'en': r'EC (mmho cm$^{-1}$)', 'fr': r'CE (mmho cm$^{-1}$)'},
    'Mg_K_Ratio_complete': {'en': 'Mg/K ratio', 'fr': 'Rapport Mg/K'},
    'Coarse_Silt_percent_complete': {'en': 'Coarse silt (%)', 'fr': 'Limon grossier (%)'},
    'Coarse_Sand_percent_complete': {'en': 'Coarse sand (%)', 'fr': 'Sable grossier (%)'},
    'growing_season_rainfall_mm': {'en': 'Growing season rain (mm)', 'fr': 'Pluie saison (mm)'},
    'CDD': {'en': 'CDD (days)', 'fr': 'CDD (jours)'},
    'SDII': {'en': r'SDII (mm day$^{-1}$)', 'fr': r'SDII (mm jour$^{-1}$)'},
    'temp_mini_mean': {'en': r'T$_{\mathrm{min}}$ (°C)', 'fr': r'T$_{\mathrm{min}}$ (°C)'},
    'temp_maxi_mean': {'en': r'T$_{\mathrm{max}}$ (°C)', 'fr': r'T$_{\mathrm{max}}$ (°C)'},
    'h_mini_mean': {'en': r'RH$_{\mathrm{min}}$ (%)', 'fr': r'HR$_{\mathrm{min}}$ (%)'},
    'h_maxi_mean': {'en': r'RH$_{\mathrm{max}}$ (%)', 'fr': r'HR$_{\mathrm{max}}$ (%)'},
    'soil_temp_6h_mean': {'en': r'T°$_{\mathrm{soil,6h}}$ (°C)', 'fr': r'T°$_{\mathrm{sol,6h}}$ (°C)'},
    'radiation_ERA_total': {'en': r'Solar rad. (cal cm$^{-2}$)', 'fr': r'Rayonnement (cal cm$^{-2}$)'},
    'CWD': {'en': 'CWD (days)', 'fr': 'CWD (jours)'},
    'Rx1': {'en': 'Rx1 (mm)', 'fr': 'Rx1 (mm)'},
    'R20': {'en': 'R20 (days)', 'fr': 'R20 (jours)'},
    'TNn': {'en': r'TN$_{\mathrm{n}}$ (°C)', 'fr': r'TN$_{\mathrm{n}}$ (°C)'},
    'TNx': {'en': r'TN$_{\mathrm{x}}$ (°C)', 'fr': r'TN$_{\mathrm{x}}$ (°C)'},
}

def get_feature_label(feature_name, lang='en'):
    """Get feature label in specified language"""
    if feature_name in FEATURE_LABELS:
        return FEATURE_LABELS[feature_name][lang]
    else:
        # Clean up feature name for display
        clean_name = feature_name.replace('_complete', '').replace('_', ' ').title()
        return clean_name

# ============================================================================
# COLOR PALETTE
# ============================================================================

VIF_COLORS = {
    'low': '#32CD32',        # Lime Green (VIF < 5)
    'moderate': '#FFD700',   # Gold (5 ≤ VIF < 10)
    'high': '#FF6B6B',       # Coral Red (VIF ≥ 10)
    'perfect': '#8B0000',    # Dark Red (VIF = ∞)
    'categorical': '#FF1493' 
}

# ============================================================================
# VIF ANALYZER CLASS
# ============================================================================

class VIFAnalyzer:
    """VIF Analyzer with bilingual support and Nature-style plotting"""
    
    def __init__(self, file_path, target_column='tc_per_ha_actual'):
        self.file_path = file_path
        self.target_column = target_column
        self.df = None
        self.vif_results = None
        self.categorical_mask = None
        self.feature_names = None
        
        print(f"{'='*80}")
        print("VIF ANALYZER FOR FEATURE SELECTION")
        print(f"{'='*80}")
    
    def load_data(self):
        """Load data from CSV"""
        print(f"\nLoading data from: {self.file_path}")
        self.df = pd.read_csv(self.file_path)
        
        if 'brix_percent' in self.df.columns:
            self.df = self.df.drop('brix_percent', axis=1)
            print("  Dropped 'brix_percent' variable")
        
        if self.target_column in self.df.columns:
            initial_count = len(self.df)
            self.df = self.df.dropna(subset=[self.target_column])
            final_count = len(self.df)
            dropped_count = initial_count - final_count
            
            if dropped_count > 0:
                print(f"  Dropped {dropped_count} observations with missing target values")
        
        print(f"  Final dataset: {self.df.shape[0]} samples, {self.df.shape[1]} columns")
    
    def prepare_features(self, selected_features, categorical_features=None, 
                        include_categorical=True):
        """Prepare features for VIF analysis with categorical support"""
        print(f"\n{'='*80}")
        print("PREPARING FEATURES FOR VIF ANALYSIS")
        print(f"{'='*80}")
        
        missing_features = [f for f in selected_features if f not in self.df.columns]
        if missing_features:
            print(f"\nWarning: Features not found in dataset:")
            for f in missing_features:
                print(f"  - {f}")
            selected_features = [f for f in selected_features if f in self.df.columns]
        
        print(f"\nSelected features provided: {len(selected_features)}")
        
        if categorical_features is None:
            categorical_features = []
            common_categorical = ['variety', 'irrigation_type', 'crop_cycle', 'harvest_mode']
            for col in common_categorical:
                if col in selected_features:
                    categorical_features.append(col)
        
        categorical_features = [f for f in categorical_features if f in selected_features]
        
        print(f"Categorical features detected: {len(categorical_features)}")
        if categorical_features:
            for f in categorical_features:
                print(f"  - {f}")
        
        numerical_features = [f for f in selected_features 
                             if f not in categorical_features 
                             and f != self.target_column]
        
        print(f"Numerical features: {len(numerical_features)}")
        
        if include_categorical and categorical_features:
            print(f"\nEncoding categorical variables with one-hot encoding...")
            
            df_numerical = self.df[numerical_features].copy()
            
            encoder = OneHotEncoder(sparse_output=False, drop='first', 
                                   handle_unknown='ignore')
            
            df_categorical = self.df[categorical_features].copy()
            for col in categorical_features:
                if df_categorical[col].isna().any():
                    mode_val = df_categorical[col].mode()[0] if not df_categorical[col].mode().empty else 'Unknown'
                    df_categorical[col] = df_categorical[col].fillna(mode_val)
                    print(f"  Filled missing values in '{col}' with mode: {mode_val}")
            
            categorical_encoded = encoder.fit_transform(df_categorical)
            
            encoded_feature_names = []
            for i, col in enumerate(categorical_features):
                categories = encoder.categories_[i][1:]
                for cat in categories:
                    encoded_feature_names.append(f"{col}_{cat}")
            
            print(f"  Created {len(encoded_feature_names)} one-hot encoded features")
            
            df_encoded_categorical = pd.DataFrame(
                categorical_encoded, 
                columns=encoded_feature_names,
                index=df_numerical.index
            )
            
            df_vif = pd.concat([df_numerical, df_encoded_categorical], axis=1)
            
            self.categorical_mask = np.array(
                [False] * len(numerical_features) + 
                [True] * len(encoded_feature_names)
            )
            
            self.feature_names = numerical_features + encoded_feature_names
            
        else:
            if categorical_features:
                print(f"\nExcluding categorical variables from analysis")
            
            df_vif = self.df[numerical_features].copy()
            self.categorical_mask = np.zeros(len(numerical_features), dtype=bool)
            self.feature_names = numerical_features
        
        for col in df_vif.columns:
            if df_vif[col].isna().any():
                median_val = df_vif[col].median()
                df_vif[col] = df_vif[col].fillna(median_val)
                print(f"  Filled missing values in '{col}' with median: {median_val:.2f}")
        
        print(f"\n{'='*80}")
        print(f"FEATURES PREPARED FOR VIF ANALYSIS")
        print(f"{'='*80}")
        print(f"Total features: {len(df_vif.columns)}")
        print(f"  - Numerical: {(~self.categorical_mask).sum()}")
        print(f"  - Categorical (encoded): {self.categorical_mask.sum()}")
        print(f"Samples: {len(df_vif)}")
        
        return df_vif
    
    def calculate_vif(self, df_vif, standardize=True):
        """Calculate VIF for all features"""
        print(f"\n{'='*80}")
        print("CALCULATING VIF VALUES")
        print(f"{'='*80}")
        
        df_calc = df_vif.copy()
        
        if standardize:
            scaler = StandardScaler()
            df_calc = pd.DataFrame(
                scaler.fit_transform(df_calc),
                columns=df_calc.columns,
                index=df_calc.index
            )
            print("Features standardized for VIF calculation")
        
        vif_data = []
        
        print(f"\nCalculating VIF for {len(df_calc.columns)} features...")
        print("-" * 80)
        
        for i, feature in enumerate(df_calc.columns, 1):
            try:
                vif_value = variance_inflation_factor(df_calc.values, i - 1)
                
                if np.isinf(vif_value):
                    category = 'Perfect'
                elif vif_value >= 10:
                    category = 'High'
                elif vif_value >= 5:
                    category = 'Moderate'
                else:
                    category = 'Low'
                
                vif_data.append({
                    'Feature': feature,
                    'VIF': vif_value,
                    'Category': category,
                    'Is_Categorical': self.categorical_mask[i - 1] if self.categorical_mask is not None else False
                })
                
                if np.isinf(vif_value):
                    print(f"  {i:3d}. {feature:45s} | VIF = ∞ (Perfect correlation)")
                else:
                    print(f"  {i:3d}. {feature:45s} | VIF = {vif_value:8.2f} | {category:8s}")
                
            except Exception as e:
                print(f"  {i:3d}. {feature:45s} | ERROR: {str(e)}")
                vif_data.append({
                    'Feature': feature,
                    'VIF': np.inf,
                    'Category': 'Perfect',
                    'Is_Categorical': self.categorical_mask[i - 1] if self.categorical_mask is not None else False
                })
        
        vif_df = pd.DataFrame(vif_data)
        vif_df = vif_df.sort_values('VIF', ascending=False).reset_index(drop=True)
        
        self.vif_results = vif_df
        
        return vif_df
    
    def create_vif_barplot_horizontal(self, output_path, lang='en', max_features=30):
        """Create SHAP-style horizontal VIF bar plot"""
        
        if self.vif_results is None:
            print("Error: No VIF results available")
            return
        
        trans = TRANSLATIONS[lang]
        vif_df = self.vif_results.copy()
        
        # Limit number of features
        n_show = min(max_features, len(vif_df))
        plot_data = vif_df.head(n_show).copy()
        
        # Replace inf with large value for plotting
        max_finite_vif = plot_data[~np.isinf(plot_data['VIF'])]['VIF'].max() if len(plot_data[~np.isinf(plot_data['VIF'])]) > 0 else 20
        plot_max = max_finite_vif * 1.2
        plot_data['VIF_plot'] = plot_data['VIF'].replace(np.inf, plot_max)
        
        # Create figure - 4 inches wide x 6 inches tall (compact)
        fig, ax = plt.subplots(figsize=(4, 6))
        
        # Get feature labels in specified language
        feature_labels = [get_feature_label(f, lang) for f in plot_data['Feature']]
        
        # Assign colors based on VIF category
        colors = []
        for _, row in plot_data.iterrows():
            if row['Is_Categorical']:
                colors.append(VIF_COLORS['categorical'])
            elif np.isinf(row['VIF']):
                colors.append(VIF_COLORS['perfect'])
            elif row['VIF'] >= 10:
                colors.append(VIF_COLORS['high'])
            elif row['VIF'] >= 5:
                colors.append(VIF_COLORS['moderate'])
            else:
                colors.append(VIF_COLORS['low'])
        

        y_pos = np.arange(len(plot_data))
        bars = ax.barh(y_pos, plot_data['VIF_plot'], color=colors, 
                      alpha=0.8, height=0.5,  # SLIM BARS
                      edgecolor='white', linewidth=0.5)
        
        # Set y-axis labels with larger font
        ax.set_yticks(y_pos)
        ax.set_yticklabels(feature_labels, fontsize=11)
        ax.invert_yaxis()
        
        # Set x-axis label with larger font
        ax.set_xlabel(trans['xlabel'], fontsize=12, fontweight='600')
        
        # Set x-axis limits - compact (40% of max VIF)
        max_vif_to_show = max(8, max_finite_vif * 0.4)
        ax.set_xlim(0, max_vif_to_show)
        
        # Add threshold lines
        if max_vif_to_show >= 5:
            ax.axvline(x=5, color=VIF_COLORS['moderate'], linestyle='--', 
                      linewidth=1.0, alpha=0.7, label=trans['threshold_low'])
        if max_vif_to_show >= 8:
            if max_vif_to_show >= 10:
                ax.axvline(x=10, color=VIF_COLORS['high'], linestyle='--', 
                          linewidth=1.0, alpha=0.7, label=trans['threshold_high'])
        
        # Add value labels on bars
        max_value = plot_data['VIF_plot'].max()
        for i, (bar, row) in enumerate(zip(bars, plot_data.itertuples())):
            width = bar.get_width()
            if np.isinf(row.VIF):
                label = '∞'
            else:
                label = f'{row.VIF:.1f}'
            
            # Position label
            if row.VIF > max_vif_to_show:
                x_pos = max_vif_to_show * 0.95
                ha_align = 'right'
            else:
                x_pos = width + max_value * 0.01
                ha_align = 'left'
                    
            ax.text(x_pos, i, label, 
                   ha=ha_align, va='center', 
                   fontsize=9, fontweight='normal')
        
        # Add title
        ax.set_title(trans['title'], fontsize=13, fontweight='600', pad=10)
        
        # Grid styling
        ax.grid(True, alpha=0.25, linestyle=':', linewidth=0.7, zorder=0)
        ax.set_axisbelow(True)
        
        # Spacing
        ax.set_ylim(-0.5, len(plot_data) - 0.5)
        
        # Legend on the right side
        if max_vif_to_show >= 5:
            legend = ax.legend(loc='center right', fontsize=11, frameon=True,
                             fancybox=False, shadow=False)
        
        # Add statistics box - TOP RIGHT CORNER (above legend)
        finite_vif = vif_df[~np.isinf(vif_df['VIF'])]['VIF']
        if len(finite_vif) > 0:
            stats_text = (
                f"{trans['mean']}: {finite_vif.mean():.1f}\n"
                f"{trans['median']}: {finite_vif.median():.1f}\n"
                f"{trans['max']}: {finite_vif.max():.1f}\n"
                f"{trans['n_features']}: {len(vif_df)}"
            )
            
            props = dict(boxstyle='round,pad=0.4', facecolor='white', 
                        edgecolor='#333333', linewidth=1.0, alpha=0.95)
            ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
                   fontsize=9, verticalalignment='top', horizontalalignment='right',
                   bbox=props, family='monospace')
        
        plt.tight_layout(pad=0.5)
        plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
        
        # Also save as PDF
        pdf_path = output_path.replace('.png', '.pdf')
        plt.savefig(pdf_path, bbox_inches='tight', facecolor='white')
        
        plt.close()
        
        print(f"  ✓ {trans['saved']}: {output_path}")
        print(f"  ✓ {trans['saved']}: {pdf_path}")
    
    def create_vif_barplot_vertical(self, output_path, lang='en', max_features=30):
        """Create SHAP-style vertical VIF bar plot with 45° rotated labels - REDUCED HEIGHT BY 1/3"""
        
        if self.vif_results is None:
            print("Error: No VIF results available")
            return
        
        trans = TRANSLATIONS[lang]
        vif_df = self.vif_results.copy()
        
        # Limit number of features
        n_show = min(max_features, len(vif_df))
        plot_data = vif_df.head(n_show).copy()
        
        # FIXED: Set maximum VIF value to 10 for plotting
        FIXED_MAX_VIF = 10
        
        # Replace inf with max value (10) and cap all values at 10
        plot_data['VIF_plot'] = plot_data['VIF'].apply(lambda x: min(x, FIXED_MAX_VIF) if not np.isinf(x) else FIXED_MAX_VIF)
        
        fig, ax = plt.subplots(figsize=(8, 3.33))
        
        # Get feature labels in specified language
        feature_labels = [get_feature_label(f, lang) for f in plot_data['Feature']]
        
        # Assign colors based on VIF category
        colors = []
        for _, row in plot_data.iterrows():
            if row['Is_Categorical']:
                colors.append(VIF_COLORS['categorical'])
            elif np.isinf(row['VIF']):
                colors.append(VIF_COLORS['perfect'])
            elif row['VIF'] >= 10:
                colors.append(VIF_COLORS['high'])
            elif row['VIF'] >= 5:
                colors.append(VIF_COLORS['moderate'])
            else:
                colors.append(VIF_COLORS['low'])
        
        x_pos = np.arange(len(plot_data))
        bars = ax.bar(x_pos, plot_data['VIF_plot'], color=colors, 
                     alpha=0.8, width=0.5,  # SLIM BARS
                     edgecolor='white', linewidth=0.5)
        
        # Set x-axis labels with 45° rotation
        ax.set_xticks(x_pos)
        ax.set_xticklabels(feature_labels, fontsize=11, rotation=45, ha='right')
        
        # Set y-axis label
        ax.set_ylabel(trans['ylabel'], fontsize=12, fontweight='600')
        
        # FIXED: Set y-axis limit to exactly 10 with custom tick breaks
        ax.set_ylim(0, FIXED_MAX_VIF)
        
        # Set custom y-axis ticks: 0, 2, 4, 6, 8, 10 (showing breaks, not all values)
        ax.set_yticks([0, 2, 4, 6, 8, 10])
        ax.set_yticklabels(['0', '2', '4', '6', '8', '10'])
        
        # Add threshold lines
        ax.axhline(y=5, color=VIF_COLORS['moderate'], linestyle='--', 
                  linewidth=1.0, alpha=0.7, label=trans['threshold_low'])
        ax.axhline(y=10, color=VIF_COLORS['high'], linestyle='--', 
                  linewidth=1.0, alpha=0.7, label=trans['threshold_high'])
        
        # Add value labels on bars
        for i, (bar, row) in enumerate(zip(bars, plot_data.itertuples())):
            height = bar.get_height()
            
            # Show actual VIF value (not capped)
            if np.isinf(row.VIF):
                label = '∞'
            else:
                label = f'{row.VIF:.1f}'
            
            # Position label above bar
            if row.VIF > FIXED_MAX_VIF:
                y_pos = FIXED_MAX_VIF * 0.95
                va_align = 'top'
            else:
                y_pos = height + FIXED_MAX_VIF * 0.01
                va_align = 'bottom'
                    
            ax.text(i, y_pos, label, 
                   ha='center', va=va_align, 
                   fontsize=9, fontweight='normal')
        
        # Add title
        ax.set_title(trans['title'], fontsize=13, fontweight='600', pad=10)
        
        # Grid styling
        ax.grid(True, alpha=0.25, linestyle=':', linewidth=0.7, zorder=0)
        ax.set_axisbelow(True)
        
        # Spacing
        ax.set_xlim(-0.5, len(plot_data) - 0.5)
        
        legend = ax.legend(loc='center right', bbox_to_anchor=(1.0, 0.35), fontsize=8, frameon=False,
                         fancybox=False, shadow=False)
        
        # Add statistics box - TOP RIGHT CORNER (above legend)
        finite_vif = vif_df[~np.isinf(vif_df['VIF'])]['VIF']
        if len(finite_vif) > 0:
            stats_text = (
                f"{trans['mean']}: {finite_vif.mean():.1f}\n"
                f"{trans['median']}: {finite_vif.median():.1f}\n"
                f"{trans['max']}: {finite_vif.max():.1f}\n"
                f"{trans['n_features']}: {len(vif_df)}"
            )
            
            props = dict(boxstyle='round,pad=0.4', facecolor='white', 
                        edgecolor='#333333', linewidth=1.0, alpha=0.95)
            ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
                   fontsize=9, verticalalignment='top', horizontalalignment='right',
                   bbox=props, family='monospace')
        
        plt.tight_layout(pad=0.5)
        plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
        
        # Also save as PDF
        pdf_path = output_path.replace('.png', '.pdf')
        plt.savefig(pdf_path, bbox_inches='tight', facecolor='white')
        
        plt.close()
        
        print(f"  ✓ {trans['saved']}: {output_path}")
        print(f"  ✓ {trans['saved']}: {pdf_path}")
    
    def interpret_results(self):
        """Interpret VIF results"""
        print(f"\n{'='*80}")
        print("VIF ANALYSIS RESULTS")
        print(f"{'='*80}")
        
        if self.vif_results is None:
            print("Error: No VIF results available")
            return
        
        vif_df = self.vif_results
        
        low_vif = vif_df[vif_df['VIF'] < 5]
        moderate_vif = vif_df[(vif_df['VIF'] >= 5) & (vif_df['VIF'] < 10)]
        high_vif = vif_df[vif_df['VIF'] >= 10]
        perfect_vif = vif_df[np.isinf(vif_df['VIF'])]
        
        print(f"\nVIF CATEGORY SUMMARY:")
        print("-" * 80)
        print(f"  🟢 Low (VIF < 5):           {len(low_vif):3d} features ({len(low_vif)/len(vif_df)*100:.1f}%)")
        print(f"  🟡 Moderate (5 ≤ VIF < 10): {len(moderate_vif):3d} features ({len(moderate_vif)/len(vif_df)*100:.1f}%)")
        print(f"  🔴 High (VIF ≥ 10):         {len(high_vif):3d} features ({len(high_vif)/len(vif_df)*100:.1f}%)")
        
        if len(perfect_vif) > 0:
            print(f"  ⛔ Perfect (VIF = ∞):        {len(perfect_vif):3d} features ({len(perfect_vif)/len(vif_df)*100:.1f}%)")
        
        finite_vif = vif_df[~np.isinf(vif_df['VIF'])]['VIF']
        if len(finite_vif) > 0:
            print(f"\nVIF STATISTICS:")
            print("-" * 80)
            print(f"  Mean:   {finite_vif.mean():8.2f}")
            print(f"  Median: {finite_vif.median():8.2f}")
            print(f"  Min:    {finite_vif.min():8.2f}")
            print(f"  Max:    {finite_vif.max():8.2f}")
        
        return low_vif, moderate_vif, high_vif
    
    def get_recommendations(self, vif_threshold=10):
        """Get feature recommendations based on VIF threshold"""
        print(f"\n{'='*80}")
        print(f"RECOMMENDATIONS (VIF Threshold = {vif_threshold})")
        print(f"{'='*80}")
        
        if self.vif_results is None:
            print("Error: No VIF results available")
            return {}
        
        vif_df = self.vif_results
        
        keep_features = vif_df[vif_df['VIF'] < vif_threshold]['Feature'].tolist()
        review_features = vif_df[vif_df['VIF'] >= vif_threshold]['Feature'].tolist()
        perfect_features = vif_df[np.isinf(vif_df['VIF'])]['Feature'].tolist()
        
        print(f"\n✅ FEATURES TO KEEP (VIF < {vif_threshold}): {len(keep_features)}")
        for i, feat in enumerate(keep_features, 1):
            vif_val = vif_df[vif_df['Feature'] == feat]['VIF'].iloc[0]
            print(f"  {i:2d}. {feat:45s} | VIF = {vif_val:6.2f}")
        
        if len(review_features) > 0:
            print(f"\n⚠️  FEATURES TO REVIEW (VIF ≥ {vif_threshold}): {len(review_features)}")
            for i, feat in enumerate(review_features, 1):
                vif_val = vif_df[vif_df['Feature'] == feat]['VIF'].iloc[0]
                if np.isinf(vif_val):
                    print(f"  {i:2d}. {feat:45s} | VIF = ∞")
                else:
                    print(f"  {i:2d}. {feat:45s} | VIF = {vif_val:6.2f}")
        
        return {
            'keep_features': keep_features,
            'review_features': review_features,
            'perfect_features': perfect_features,
            'n_keep': len(keep_features),
            'n_review': len(review_features),
            'n_perfect': len(perfect_features)
        }
    
    def save_results(self, output_dir='./vif_results'):
        """Save VIF analysis results"""
        os.makedirs(output_dir, exist_ok=True)
        
        if self.vif_results is None:
            print("Error: No VIF results to save")
            return
        
        print(f"\n{'='*80}")
        print("SAVING VIF ANALYSIS RESULTS")
        print(f"{'='*80}")
        
        vif_path = os.path.join(output_dir, 'vif_analysis_results.csv')
        self.vif_results.to_csv(vif_path, index=False)
        print(f"  Saved: {vif_path}")
        
        recommended = self.vif_results[self.vif_results['VIF'] < 10]['Feature'].tolist()
        rec_path = os.path.join(output_dir, 'recommended_features_vif_below_10.csv')
        pd.DataFrame({'Feature': recommended}).to_csv(rec_path, index=False)
        print(f"  Saved: {rec_path}")

# ============================================================================
# MAIN EXECUTION FUNCTION
# ============================================================================

def run_vif_analysis(file_path, selected_features, 
                    categorical_features=None,
                    include_categorical=True,
                    vif_threshold=10,
                    create_plots=True,
                    save_results=True,
                    output_dir=None,
                    max_features_plot=30):
    """
    Main function to run VIF analysis
    
    Parameters:
    -----------
    file_path : str
        Path to CSV file
    selected_features : list
        List of features to analyze
    categorical_features : list, optional
        List of categorical features
    include_categorical : bool
        Include categorical variables (one-hot encoded)
    vif_threshold : float
        VIF threshold for recommendations
    create_plots : bool
        Create bar plots (both horizontal and vertical)
    save_results : bool
        Save results to files
    output_dir : str, optional
        Output directory (if None, creates 'vif_results' folder next to data file)
    max_features_plot : int
        Maximum features to show in plot
    
    Returns:
    --------
    dict : Analysis results
    """
    print(f"\n{'#'*80}")
    print("VIF ANALYSIS FOR FEATURE SELECTION")
    print(f"{'#'*80}\n")
    
    # Set output directory to be in the same folder as the data file
    if output_dir is None:
        data_dir = os.path.dirname(os.path.abspath(file_path))
        output_dir = os.path.join(data_dir, 'vif_results')
    
    print(f"Output directory: {output_dir}\n")
    
    # Initialize analyzer
    analyzer = VIFAnalyzer(file_path)
    
    # Load data
    analyzer.load_data()
    
    # Prepare features
    df_vif = analyzer.prepare_features(
        selected_features=selected_features,
        categorical_features=categorical_features,
        include_categorical=include_categorical
    )
    
    if len(df_vif.columns) == 0:
        print("\nError: No features available for VIF analysis")
        return None
    
    if len(df_vif.columns) == 1:
        print("\nWarning: Only one feature available. VIF requires multiple features.")
        return None
    
    # Calculate VIF
    vif_df = analyzer.calculate_vif(df_vif, standardize=True)
    
    # Interpret results
    low_vif, moderate_vif, high_vif = analyzer.interpret_results()
    
    # Get recommendations
    recommendations = analyzer.get_recommendations(vif_threshold=vif_threshold)
    
    # Create plots for both languages (horizontal and vertical)
    if create_plots:
        os.makedirs(output_dir, exist_ok=True)
        
        print(f"\n{'='*80}")
        print("CREATING VIF BAR PLOTS (HORIZONTAL & VERTICAL)")
        print(f"{'='*80}")
        
        # HORIZONTAL PLOTS
        print("\n--- HORIZONTAL BAR PLOTS ---")
        
        # English horizontal plot
        print("\nGenerating English horizontal plot...")
        en_h_path = os.path.join(output_dir, 'vif_barplot_horizontal_en.png')
        analyzer.create_vif_barplot_horizontal(en_h_path, lang='en', max_features=max_features_plot)
        
        # French horizontal plot
        print("\nGenerating French horizontal plot...")
        fr_h_path = os.path.join(output_dir, 'vif_barplot_horizontal_fr.png')
        analyzer.create_vif_barplot_horizontal(fr_h_path, lang='fr', max_features=max_features_plot)
        
        # VERTICAL PLOTS
        print("\n--- VERTICAL BAR PLOTS ---")
        
        # English vertical plot
        print("\nGenerating English vertical plot...")
        en_v_path = os.path.join(output_dir, 'vif_barplot_vertical_en.png')
        analyzer.create_vif_barplot_vertical(en_v_path, lang='en', max_features=max_features_plot)
        
        # French vertical plot
        print("\nGenerating French vertical plot...")
        fr_v_path = os.path.join(output_dir, 'vif_barplot_vertical_fr.png')
        analyzer.create_vif_barplot_vertical(fr_v_path, lang='fr', max_features=max_features_plot)
    
    # Save results
    if save_results:
        analyzer.save_results(output_dir=output_dir)
    
    # Final summary
    print(f"\n{'#'*80}")
    print("VIF ANALYSIS COMPLETE")
    print(f"{'#'*80}")
    print(f"\nSummary:")
    print(f"  Total features analyzed:     {len(vif_df)}")
    print(f"  Low VIF (< 5):              {len(low_vif)} ({len(low_vif)/len(vif_df)*100:.1f}%)")
    print(f"  Moderate VIF (5-10):        {len(moderate_vif)} ({len(moderate_vif)/len(vif_df)*100:.1f}%)")
    print(f"  High VIF (≥ 10):            {len(high_vif)} ({len(high_vif)/len(vif_df)*100:.1f}%)")
    print(f"  Recommended (< {vif_threshold}):        {recommendations['n_keep']}")
    print(f"\nOutput location:")
    print(f"  • {output_dir}")
    
    return {
        'analyzer': analyzer,
        'vif_results': vif_df,
        'recommendations': recommendations,
        'low_vif': low_vif,
        'moderate_vif': moderate_vif,
        'high_vif': high_vif,
        'output_dir': output_dir
    }

# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    
    print(f"\n{'='*80}")
    print("VIF ANALYSIS WITH HORIZONTAL & VERTICAL BAR PLOTS")
    print(f"{'='*80}\n")
    
    # Define selected features
    selected_features = [
        'pH_complete', 'Exchangeable_Calcium_meq_100g_complete', 
        'Exchangeable_Potassium_meq_100g_complete', 'Exchangeable_Sodium_meq_100g_complete',
        'Available_P2O5_mg_kg_complete', 'Total_Nitrogen_per_mille_complete', 
        'Electrical_Conductivity_mmho_cm_complete', 'Mg_K_Ratio_complete',
        'Coarse_Silt_percent_complete', 'Coarse_Sand_percent_complete', 
        'growing_season_rainfall_mm', 'CDD', 'SDII',
        'temp_mini_mean', 'temp_maxi_mean', 'h_mini_mean', 'h_maxi_mean',
        'soil_temp_6h_mean', 'radiation_ERA_total', 'CWD', 'Rx1', 'R20', 'TNn', 'TNx'
    ]
    
    # Define categorical features
    categorical_features = ['variety', 'irrigation_type', 'crop_cycle', 'harvest_mode']
    categorical_in_selection = [f for f in categorical_features if f in selected_features]
    
    # Run VIF analysis
    try:
        results = run_vif_analysis(
            file_path='C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv',
            selected_features=selected_features,
            categorical_features=categorical_in_selection,
            include_categorical=True,
            vif_threshold=10,
            create_plots=True,
            save_results=True,
            # output_dir will be auto-created in same folder as data file
            max_features_plot=30
        )
        
        if results:
            print(f"\n{'='*80}")
            print("FINAL RECOMMENDATIONS")
            print(f"{'='*80}")
            
            recommended = results['recommendations']['keep_features']
            
            print(f"\n✅ USE THESE {len(recommended)} FEATURES:")
            for i, feat in enumerate(recommended, 1):
                vif_val = results['vif_results'][results['vif_results']['Feature'] == feat]['VIF'].iloc[0]
                print(f"  {i:2d}. {feat:45s} | VIF = {vif_val:6.2f}")
        
    except FileNotFoundError:
        print(f"\nError: Data file not found.")
        print("Please update the file_path in the code.")
    
    except Exception as e:
        print(f"\nAn error occurred: {str(e)}")
        import traceback
        traceback.print_exc()
    
    print(f"\n{'='*80}")
    print("VIF ANALYSIS COMPLETE")
    print(f"{'='*80}")


VIF ANALYSIS WITH HORIZONTAL & VERTICAL BAR PLOTS


################################################################################
VIF ANALYSIS FOR FEATURE SELECTION
################################################################################

Output directory: C:\Users\U078-N800\Desktop\Ml_Internship\Copy\Ml_Training\Regrouped_Analysis\vif_results

VIF ANALYZER FOR FEATURE SELECTION

Loading data from: C:/Users/U078-N800/Desktop/Ml_Internship/Copy/Ml_Training/Regrouped_Analysis/final_dataset_regrouped_categorical.csv
  Dropped 'brix_percent' variable
  Final dataset: 13376 samples, 67 columns

PREPARING FEATURES FOR VIF ANALYSIS

Selected features provided: 24
Categorical features detected: 0
Numerical features: 24

FEATURES PREPARED FOR VIF ANALYSIS
Total features: 24
  - Numerical: 24
  - Categorical (encoded): 0
Samples: 13376

CALCULATING VIF VALUES
Features standardized for VIF calculation

Calculating VIF for 24 features...
------------------------------------------------